# Time Series Track · ML Feature Engineering for Time Series

**Track:** Traditional AI Domains — Time Series  
**Prerequisites:** TS/01 (Classical Forecasting), ML/03 (Tree Models)  
**Difficulty:** ⭐⭐ Intermediate  
**Estimated Time:** 90–120 minutes

---

### 🔍 Socratic Deep Check
*How do you transform a chronological sequence into a tabular spreadsheet that LightGBM — which has absolutely no concept of "time" — can use to out-predict ARIMA?*

---

### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Classical models (ARIMA, Prophet) naturally understand sequences. Machine Learning models (XGBoost, Random Forests) are completely blind to sequences — they only understand rows and columns. The entire art of applying ML to time series lies in **Feature Engineering**: transforming time structures into columns (lags, rolling means, sine/cosine waves) so gradient-boosted trees can find complex, non-linear interactions.

**Mental Model:** Imagine translating a movie (time series) into a series of photographs (tabular data). To predict what happens in photo $T$, you must artificially attach data from photo $T-1$, $T-7$, and $T-30$ directly onto the metadata of photo $T$. 

**Common Junior Pitfall:** Feature Leakage via standard cross-validation. Juniors will use `sklearn.model_selection.KFold` or `train_test_split(random_state=42)` on time series features. This allows the model to train on Friday's data to predict Thursday's metric. Time Series requires strict **Temporal Cross-Validation** (`TimeSeriesSplit`) where the validation set is always strictly in the future relative to the training set.

---


## 📑 Table of Contents

* [1 · Flattening Time: Feature Engineering Pipeline](#1--flattening-time-feature-engineering-pipeline)
  * [Lags, Rolling Stats, and Calendars](#lags-rolling-stats-and-calendars)
* [2 · The Danger of Random Splitting](#2--the-danger-of-random-splitting)
  * [Implementing TimeSeriesSplit](#implementing-timeseriessplit)
* [3 · Multi-Step Forecasting Strategies](#3--multi-step-forecasting-strategies)
  * [Recursive vs Direct Forecasting](#recursive-vs-direct-forecasting)
* [✅ Knowledge Check](#-knowledge-check)
* [🔨 Practical Practice](#-practical-practice)
* [🎯 Summary & Key Takeaways](#-summary--key-takeaways)


---
## 1 · Flattening Time: Feature Engineering Pipeline

To use XGBoost/LightGBM, we must build three classes of features:
1. **Lag Features:** Past values ($y_{t-1}, y_{t-7}$).
2. **Window Statistics:** Moving averages to capture trend momentum.
3. **Calendar/Fourier Features:** Encoding cyclical seasonality without relying on sequential order.


In [ ]:
# Cell 1 — Create time series feature engineering pipeline
import pandas as pd
import numpy as np

# 1. Generate Synthetic Retail Data
np.random.seed(42)
n = 365 * 3
dates = pd.date_range('2022-01-01', periods=n, freq='D')

# Components: Trend + Yearly Seasonality + Weekly pattern + Noise
trend = np.linspace(100, 180, n)
season = 20 * np.sin(2 * np.pi * np.arange(n) / 365)
weekly = 15 * np.sin(2 * np.pi * np.arange(n) / 7)
noise = np.random.normal(0, 5, n)
values = trend + season + weekly + noise + np.where(np.arange(n) > 600, 30, 0) # level shift

df = pd.DataFrame({'date': dates, 'revenue': values})
df = df.set_index('date')

def create_ts_features(df, target_col='revenue', lags=[1,7,14,28], windows=[7,14,30]):
    """Transform time series into ML-ready feature table.
    ALL features look BACKWARD only — preventing data leakage.
    """
    result = df.copy()
    
    # 1. LAG features (past values as absolute predictors)
    for lag in lags:
        result[f'lag_{lag}'] = result[target_col].shift(lag)
    
    # 2. ROLLING statistics (smoothed trends)
    # Note: We shift by 1 BEFORE rolling so today's value isn't included in its own rolling mean
    for w in windows:
        result[f'rolling_mean_{w}'] = result[target_col].shift(1).rolling(w).mean()
        result[f'rolling_std_{w}'] = result[target_col].shift(1).rolling(w).std()
    
    # 3. CALENDAR features (encode cyclical patterns)
    result['day_of_week'] = result.index.dayofweek
    result['day_of_year'] = result.index.dayofyear
    result['is_weekend'] = (result.index.dayofweek >= 5).astype(int)
    
    # 4. FOURIER terms (smooth continuous seasonality for tree models)
    # Trees naturally do threshold splits. Sine waves help them map smooth cycles.
    for k in [1, 2, 3]:
        result[f'sin_yearly_{k}'] = np.sin(2 * np.pi * k * result.index.dayofyear / 365)
        result[f'cos_yearly_{k}'] = np.cos(2 * np.pi * k * result.index.dayofyear / 365)
    
    return result.dropna()

df_features = create_ts_features(df)
print(f'Feature table: {df_features.shape[0]} rows × {df_features.shape[1]} columns')
print(f'Sample engineered row:\n{df_features.iloc[50].to_dict()}')


---
## 2 · The Danger of Random Splitting

If you use classical K-Fold cross validation on temporal tabular data, the model will essentially "cheat" by learning from the future to predict the past. We must use **Temporal Backtesting**.


In [ ]:
# Cell 2 — TimeSeries Validation Demonstration
from sklearn.model_selection import TimeSeriesSplit, KFold
import matplotlib.pyplot as plt

def plot_cv_splits(cv_obj, X, y, ax, title):
    """Visualize CV splits to show leakage mechanics."""
    for ii, (tr, tt) in enumerate(cv_obj.split(X=X, y=y)):
        p1 = ax.scatter(tr, [ii] * len(tr), c='steelblue', marker='_', lw=8)
        p2 = ax.scatter(tt, [ii] * len(tt), c='firebrick', marker='_', lw=8)
    ax.set_title(title)
    ax.set_ylabel('CV Iteration')
    ax.set_xlabel('Time Step Index')
    ax.legend([p1, p2], ['Training Data', 'Validation Data'], loc='upper right')

fig, ax = plt.subplots(1, 2, figsize=(15, 4))

# 1. The DANGEROUS way (Random KFold)
bad_cv = KFold(n_splits=5, shuffle=True, random_state=42)
plot_cv_splits(bad_cv, df_features, df_features['revenue'], ax[0], 'Catastrophic Leakage: Shuffled K-Fold')
ax[0].text(x=200, y=2.5, s="Validation targets mix past & future!", color='black', alpha=0.9)

# 2. The CORRECT way (TimeSeriesSplit)
good_cv = TimeSeriesSplit(n_splits=5)
plot_cv_splits(good_cv, df_features, df_features['revenue'], ax[1], 'Correct Execution: TimeSeriesSplit')
plt.tight_layout()
plt.show()


---
## 3 · Multi-Step Forecasting Strategies

XGBoost natively predicts exactly ONE step ahead. If you want to predict the next 7 days, you must choose an architectural strategy:

| Strategy | How it Works | Pros | Cons |
|----------|--------------|------|------|
| **Recursive (Autoregressive)** | Predict $T+1$. Append prediction to data as `lag_1`. Predict $T+2$. | Needs only 1 model. Fits dynamic paths. | Errors compound exponentially. By $T+7$, it's drifting on its own errors. |
| **Direct Objective** | Train 7 separate XGBoost models. Model 1 predicts $Y_{t+1}$, Model 7 predicts $Y_{t+7}$. | No error compounding! Highly accurate for endpoints. | Massive compute footprint (requires N models). Cannot capture cross-step dynamics. |
| **Multi-Output** | Train 1 model to yield a length-7 vector simultaneously. | One model architecture. | Hard to optimize tree structures for multi-scalar targets natively. |


In [ ]:
# Cell 3 — Training an XGBoost Direct/Recursive hybrid logic
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error

# Split context
target = 'revenue'
features = [c for c in df_features.columns if c != target]

train_idx = int(len(df_features) * 0.8)
train = df_features.iloc[:train_idx]
test = df_features.iloc[train_idx:]

X_train, y_train = train[features], train[target]
X_test, y_test = test[features], test[target]

# Fit Base Model
xgb = XGBRegressor(n_estimators=150, max_depth=5, learning_rate=0.08, verbosity=0)
xgb.fit(X_train, y_train)

# Basic 1-Step Validation
xgb_pred = xgb.predict(X_test)
xgb_mae = mean_absolute_error(y_test, xgb_pred)
print(f"XGBoost Test MAE (1-step horizon): {xgb_mae:.3f}")

# Feature Importance
importance = pd.Series(xgb.feature_importances_, index=features).sort_values(ascending=False)
print(f'\nTop 7 Explanatory Features for Revenue:')
for feat, imp in importance.head(7).items():
    print(f'  {feat:20s}: {imp:.4f}')


---
## ✅ Knowledge Check

### Q1: Leakage in Shift Operator
Why does using `df['target'].shift(-1)` to generate table features instantly sabotage the time series ML pipeline?
<details><summary>Answer</summary>
`shift(-1)` pulls the value from the future row into the current row. The model will literally "learn" that tomorrow's revenue predicts tomorrow's revenue perfectly. In production, you don't possess tomorrow's revenue, so the inference matrix will be full of nulls, crashing or yielding absurd predictions.
</details>

### Q2: Fourier Transformations
Why are `sin()` and `cos()` transformations of the `day_of_year` useful for XGBoost/LightGBM?
<details><summary>Answer</summary>
Tree algorithms perform hard orthogonal axis-splits (e.g. `day_of_year <= 340`). This forces them to learn jagged step-functions to approximate seasonal curves, and they completely fail to realize that December 31st (365) and January 1st (1) are adjacent in time. Fourier terms map the days onto a continuous cyclical dimension where Day 365 and Day 1 sit seamlessly next to each other.
</details>

---
## 🔨 Practical Practice

### Exercise 1: Advanced Calendar Engineering
Extend the `create_ts_features` function to incorporate US Holiday mappings (using the Python `holidays` package). Convert it to a binary boolean array.

### Exercise 2: Implementing Recursive State
Write a recursive loop that predicts the next 7 days without looking at `y_test`. Instead of using actual lags, use `.iloc[-1]` from your running prediction list and slide your window manually to construct the synthetic `X_test` input tensors on the fly.


---
## 🎯 Summary & Key Takeaways

1. **Tabular Reduction:** ML models require time to be "flattened" onto the horizontal row via highly engineered lagging.
2. **Strict Causality:** Future data must NEVER leak into the past rows. All engineering operations must be strictly historical shifts.
3. **Temporal Validation:** Random K-Fold ruins temporal evaluation. Always use `TimeSeriesSplit` to mimic a sliding chronological window.
4. **Extrapolation Limits:** Gradient boosted trees CANNOT extrapolate trendlines into numbers they've never seen in the training domain. If revenue is going linearly to infinity, XGBoost will hit an asymptotic ceiling unless you difference ($Y_{t} - Y_{t-1}$) the target to make it stationary.

**Next →** `03_deep_learning_time_series.ipynb` — Transitioning to native sequence processing with LSTMs, Temporal Convolutional Networks (TCN), and Time-Series Transformers.
